# B1 — Meta-Initialized Self-Supervised TTT (MI-S3T)

FOMAML çerçevesi + masked LM inner loop. Eğitimde her batch support/query'ye bölünür, outer Adam adımı orijinal ağırlık üzerinde uygulanır.

**Bağımsız çalışır.** Tüm parametreler Hücre 2'de toplanmıştır — sadece o hücreyi düzenleyerek deneyi kontrol edebilirsin. Sonuçlar `./results/B1_MI_S3T_*` altına kaydedilir.

In [ ]:
# ─── HÜCRE 1: Setup ────────────────────────────────────────────────────
# Colab tespit + pip install + Drive mount (opsiyonel) + arf_lib import
import os, sys, importlib
IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    # Colab'da arf_lib.py'nin yerini bul — varsayım:
    # repo /content/CALMA/standalone_notebooks/ altında klonlu
    REPO_BASE = "/content/CALMA"
    NB_DIR = f"{REPO_BASE}/standalone_notebooks"
    if not os.path.exists(NB_DIR):
        !git clone https://github.com/hakanskn/CALMA.git {REPO_BASE} 2>&1 | tail -5
    sys.path.insert(0, NB_DIR)

    # Drive mount (sonuçları Drive'a yedeklemek istersen PARAMS'ta results_root değiştir)
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    except Exception as e:
        print("Drive mount skipped:", e)

    !pip install -q transformers==4.41.0 datasets==2.19.0 tokenizers==0.19.0 accelerate==0.30.0 matplotlib seaborn
else:
    # Lokal: arf_lib.py notebook ile aynı klasörde
    NB_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
    sys.path.insert(0, NB_DIR)

import arf_lib
importlib.reload(arf_lib)
print(f"arf_lib loaded from: {arf_lib.__file__}")
print(f"GPU info: {arf_lib.gpu_info()}")

In [ ]:
# ─── HÜCRE 2: PARAMETRELER ─────────────────────────────────────────────
PARAMS = {
    "method_name":   "B1_MI_S3T",
    "run_name":      "wt2_seed42",
    "seed":          42,
    "results_root":  "./results",

    "model_name":    "gpt2",
    "tokenizer_name": "gpt2",

    "dataset":       "wikitext2",
    "max_seq_len":   512,
    "batch_size":    16,
    "num_workers":   2,

    "learning_rate": 5e-5,      # outer optimizer
    "num_epochs":    3,
    "warmup_steps":  0,         # MAML çerçevesi: scheduler yok
    "grad_clip":     1.0,
    "weight_decay":  0.01,

    "limit_train_batches": 100,
    "limit_eval_batches":  50,

    "log_every_n_steps":        25,
    "save_checkpoints":         False,
    "checkpoint_every_n_steps": 500,
    "keep_last_n_checkpoints":  3,

    # ─ Method-specific (MI-S3T)
    "mi_inner_lr":     1e-4,        # inner SGD lr
    "mi_inner_steps":  3,           # inner adım sayısı
    "mi_outer_lr":     5e-5,        # outer Adam lr
    "mi_mask_ratio":   0.15,        # masked LM oranı
    "mi_use_fomaml":   True,
}

In [ ]:
# ─── HÜCRE 3: Imports ──────────────────────────────────────────────────
import math, time, json
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from arf_lib import (
    seed_everything, get_device, gpu_info, gpu_peak_mb,
    GPT2Wrapper, BaseRouterAttention,
    StandaloneTrainer, load_text_dataloaders,
    run_pipeline, save_results, append_to_index, make_plots,
)

seed_everything(PARAMS["seed"])
print(f"Seed set: {PARAMS['seed']}")
print(f"Device: {get_device()}")

In [ ]:
# ─── HÜCRE 4: Method — Meta-Initialized Self-Supervised TTT ────────────
def _mask_tokens(input_ids, mask_token_id, mask_ratio=0.15, ignore_index=-100):
    input_ids = input_ids.clone()
    labels = input_ids.clone()
    probs = torch.full(input_ids.shape, mask_ratio, device=input_ids.device)
    mask = torch.bernoulli(probs).bool()
    labels[~mask] = ignore_index
    input_ids[mask] = mask_token_id
    return input_ids, labels


class MISSelfSupervisedTTT:
    requires_meta_loop = True
    owns_optimizer = True

    def __init__(self, model, params):
        self.model = model
        mp = params
        self.inner_lr = float(mp.get("mi_inner_lr", 1e-4))
        self.inner_steps = int(mp.get("mi_inner_steps", 3))
        self.outer_lr = float(mp.get("mi_outer_lr", 5e-5))
        self.mask_ratio = float(mp.get("mi_mask_ratio", 0.15))
        self.mask_token_id = 50256
        self.outer_optimizer = torch.optim.Adam(self.model.parameters(), lr=self.outer_lr)

    def set_mask_token_id(self, tid):
        self.mask_token_id = tid

    def _ss_loss(self, batch):
        masked, labels = _mask_tokens(batch["input_ids"], self.mask_token_id, self.mask_ratio)
        out = self.model(input_ids=masked, attention_mask=batch.get("attention_mask"), labels=labels)
        return out.loss

    def adapt(self, context):
        was = self.model.training; self.model.train()
        inner_opt = torch.optim.SGD(self.model.parameters(), lr=self.inner_lr)
        for _ in range(self.inner_steps):
            loss = self._ss_loss(context)
            inner_opt.zero_grad(); loss.backward(); inner_opt.step()
        if not was: self.model.eval()

    def meta_train_step(self, support, query):
        original = {n: p.data.clone() for n, p in self.model.named_parameters()}
        self.adapt(support)
        q_loss = self._ss_loss(query)
        self.outer_optimizer.zero_grad()
        q_loss.backward()
        for n, p in self.model.named_parameters():
            p.data.copy_(original[n])
        self.outer_optimizer.step()
        return float(q_loss.item())

print("MISSelfSupervisedTTT tanımlandı.")

In [ ]:
def build_model_and_adapter(params):
    from transformers import GPT2LMHeadModel
    model = GPT2LMHeadModel.from_pretrained(params["model_name"])
    adapter = MISSelfSupervisedTTT(model, params)
    print(f"MI-S3T adapter → {sum(p.numel() for p in model.parameters()):,} params")
    return model, adapter

In [ ]:
# ─── HÜCRE 6: Run ──────────────────────────────────────────────────────
# Tek satır: model + opsiyonel adapter inşa et, pipeline çalıştır
model, adapter = build_model_and_adapter(PARAMS)
run_dir, result = run_pipeline(model, PARAMS, adapter=adapter)
fm = result["final_metrics"]
print("\n" + "=" * 60)
print(f"RUN DIR: {run_dir}")
print(f"Final test PPL:             {fm['test_ppl']:.4f}")
print(f"Final test BPC (per-char):  {fm['test_bpc']:.4f}")
print(f"Final test bits-per-token:  {fm['test_bits_per_token']:.4f}")
print(f"chars/token:                {fm['chars_per_token']:.3f}")
print(f"Duration: {result['duration_seconds']:.1f}s")
print("=" * 60)

In [ ]:
# ─── HÜCRE 7: Display plots (inline) ───────────────────────────────────
from IPython.display import Image, display
import os
plots_dir = run_dir / "plots"
if plots_dir.exists():
    for p in sorted(plots_dir.glob("*.png")):
        print(f"📈 {p.name}")
        display(Image(filename=str(p)))
else:
    print("Plots not generated.")
print(f"\nAll outputs in: {run_dir}")
print(f"Index file: {Path(PARAMS['results_root']) / '_index.csv'}")